In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google.cloud import bigquery
import pandas as pd

# Load environment variables (sets GOOGLE_APPLICATION_CREDENTIALS for the BigQuery client)
load_dotenv(dotenv_path=r"D:\Shikha\bias_project\.env")

# Verify the credentials path is set
creds_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
print(f"Credentials path: {creds_path}")
print(f"Credentials file exists: {Path(creds_path).exists() if creds_path else False}")

# Initialize BigQuery client
client = bigquery.Client(project="bias-project-gdelt")
print(f"\nBigQuery client initialized")
print(f"Project: {client.project}")

# Test query — count GDELT GKG records for one day to confirm access
test_query = """
SELECT COUNT(*) as record_count
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME = TIMESTAMP("2024-01-15")
"""

print(f"\nRunning test query...")
result = client.query(test_query).to_dataframe()
print(f"Records for 2024-01-15: {result['record_count'].iloc[0]:,}")

Credentials path: D:\Shikha\bias_project\gcp-key.json
Credentials file exists: True

BigQuery client initialized
Project: bias-project-gdelt

Running test query...


d:\Shikha\bias_project\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Records for 2024-01-15: 378,373


In [2]:
# Build a mapping of US newspaper domains to their states
# Start with major dailies — can be expanded later

US_NEWSPAPER_DOMAINS = {
    # === Northeast ===
    'nytimes.com':              ('NY', 'New York Times', 'national'),
    'nypost.com':               ('NY', 'New York Post', 'large'),
    'nydailynews.com':          ('NY', 'New York Daily News', 'large'),
    'newsday.com':              ('NY', 'Newsday', 'medium'),
    'buffalonews.com':          ('NY', 'Buffalo News', 'medium'),
    'syracuse.com':             ('NY', 'Syracuse Post-Standard', 'medium'),
    'democratandchronicle.com': ('NY', 'Rochester Democrat and Chronicle', 'medium'),
    'washingtonpost.com':       ('DC', 'Washington Post', 'national'),
    'baltimoresun.com':         ('MD', 'Baltimore Sun', 'large'),
    'inquirer.com':             ('PA', 'Philadelphia Inquirer', 'large'),
    'post-gazette.com':         ('PA', 'Pittsburgh Post-Gazette', 'large'),
    'pennlive.com':             ('PA', 'Patriot-News (Harrisburg)', 'medium'),
    'mcall.com':                ('PA', 'Allentown Morning Call', 'medium'),
    'bostonglobe.com':          ('MA', 'Boston Globe', 'large'),
    'bostonherald.com':         ('MA', 'Boston Herald', 'medium'),
    'masslive.com':             ('MA', 'MassLive (Springfield Republican)', 'medium'),
    'hartfordcourant.com':      ('CT', 'Hartford Courant', 'medium'),
    'ctpost.com':               ('CT', 'Connecticut Post', 'medium'),
    'nj.com':                   ('NJ', 'New Jersey Star-Ledger', 'large'),
    'northjersey.com':          ('NJ', 'The Record (Bergen)', 'medium'),
    'app.com':                  ('NJ', 'Asbury Park Press', 'medium'),
    'providencejournal.com':    ('RI', 'Providence Journal', 'medium'),
    'pressherald.com':          ('ME', 'Portland Press Herald', 'medium'),
    'unionleader.com':          ('NH', 'Union Leader (Manchester)', 'medium'),
    'burlingtonfreepress.com':  ('VT', 'Burlington Free Press', 'small'),
    
    # === Midwest ===
    'chicagotribune.com':       ('IL', 'Chicago Tribune', 'large'),
    'suntimes.com':             ('IL', 'Chicago Sun-Times', 'large'),
    'pjstar.com':               ('IL', 'Peoria Journal Star', 'small'),
    'sj-r.com':                 ('IL', 'Springfield State Journal-Register', 'small'),
    'detroitnews.com':          ('MI', 'Detroit News', 'large'),
    'freep.com':                ('MI', 'Detroit Free Press', 'large'),
    'mlive.com':                ('MI', 'MLive (Grand Rapids, etc.)', 'medium'),
    'cleveland.com':            ('OH', 'Cleveland Plain Dealer', 'large'),
    'dispatch.com':             ('OH', 'Columbus Dispatch', 'large'),
    'cincinnati.com':           ('OH', 'Cincinnati Enquirer', 'medium'),
    'daytondailynews.com':      ('OH', 'Dayton Daily News', 'medium'),
    'startribune.com':          ('MN', 'Minneapolis Star Tribune', 'large'),
    'twincities.com':           ('MN', 'St. Paul Pioneer Press', 'medium'),
    'jsonline.com':             ('WI', 'Milwaukee Journal Sentinel', 'large'),
    'madison.com':              ('WI', 'Wisconsin State Journal (Madison)', 'medium'),
    'stltoday.com':             ('MO', 'St. Louis Post-Dispatch', 'large'),
    'kansascity.com':           ('MO', 'Kansas City Star', 'large'),
    'desmoinesregister.com':    ('IA', 'Des Moines Register', 'medium'),
    'omaha.com':                ('NE', 'Omaha World-Herald', 'medium'),
    'kansas.com':               ('KS', 'Wichita Eagle', 'medium'),
    'argusleader.com':          ('SD', 'Sioux Falls Argus Leader', 'small'),
    'bismarcktribune.com':      ('ND', 'Bismarck Tribune', 'small'),
    'indystar.com':             ('IN', 'Indianapolis Star', 'large'),
    'jconline.com':             ('IN', 'Lafayette Journal & Courier', 'small'),
    
    # === South ===
    'ajc.com':                  ('GA', 'Atlanta Journal-Constitution', 'large'),
    'savannahnow.com':          ('GA', 'Savannah Morning News', 'medium'),
    'macon.com':                ('GA', 'Macon Telegraph', 'small'),
    'tampabay.com':             ('FL', 'Tampa Bay Times', 'large'),
    'miamiherald.com':          ('FL', 'Miami Herald', 'large'),
    'orlandosentinel.com':      ('FL', 'Orlando Sentinel', 'large'),
    'sun-sentinel.com':         ('FL', 'South Florida Sun-Sentinel', 'medium'),
    'jacksonville.com':         ('FL', 'Florida Times-Union (Jacksonville)', 'medium'),
    'palmbeachpost.com':        ('FL', 'Palm Beach Post', 'medium'),
    'al.com':                   ('AL', 'AL.com (Birmingham News)', 'medium'),
    'montgomeryadvertiser.com': ('AL', 'Montgomery Advertiser', 'small'),
    'clarionledger.com':        ('MS', 'Clarion-Ledger (Jackson)', 'medium'),
    'sunherald.com':            ('MS', 'Biloxi Sun Herald', 'small'),
    'nola.com':                 ('LA', 'Times-Picayune (New Orleans)', 'large'),
    'shreveporttimes.com':      ('LA', 'Shreveport Times', 'small'),
    'tennessean.com':           ('TN', 'The Tennessean (Nashville)', 'large'),
    'commercialappeal.com':     ('TN', 'Memphis Commercial Appeal', 'medium'),
    'knoxnews.com':             ('TN', 'Knoxville News Sentinel', 'medium'),
    'jacksonsun.com':           ('TN', 'Jackson Sun', 'small'),
    'courier-journal.com':      ('KY', 'Louisville Courier-Journal', 'medium'),
    'kentucky.com':             ('KY', 'Lexington Herald-Leader', 'medium'),
    'newsobserver.com':         ('NC', 'Raleigh News & Observer', 'large'),
    'charlotteobserver.com':    ('NC', 'Charlotte Observer', 'large'),
    'wral.com':                 ('NC', 'WRAL Raleigh', 'medium'),
    'thestate.com':             ('SC', 'The State (Columbia)', 'medium'),
    'postandcourier.com':       ('SC', 'Charleston Post and Courier', 'medium'),
    'richmond.com':             ('VA', 'Richmond Times-Dispatch', 'large'),
    'pilotonline.com':          ('VA', 'Virginian-Pilot', 'medium'),
    'roanoke.com':              ('VA', 'Roanoke Times', 'small'),
    'wvgazettemail.com':        ('WV', 'Charleston Gazette-Mail', 'small'),
    'arkansasonline.com':       ('AR', 'Arkansas Democrat-Gazette', 'medium'),
    'dallasnews.com':           ('TX', 'Dallas Morning News', 'large'),
    'star-telegram.com':        ('TX', 'Fort Worth Star-Telegram', 'large'),
    'houstonchronicle.com':     ('TX', 'Houston Chronicle', 'large'),
    'chron.com':                ('TX', 'Houston Chronicle (chron)', 'large'),
    'expressnews.com':          ('TX', 'San Antonio Express-News', 'large'),
    'statesman.com':            ('TX', 'Austin American-Statesman', 'medium'),
    'elpasotimes.com':          ('TX', 'El Paso Times', 'medium'),
    'oklahoman.com':            ('OK', 'The Oklahoman', 'medium'),
    'tulsaworld.com':           ('OK', 'Tulsa World', 'medium'),
    
    # === West ===
    'latimes.com':              ('CA', 'Los Angeles Times', 'national'),
    'sfchronicle.com':          ('CA', 'San Francisco Chronicle', 'large'),
    'sfgate.com':               ('CA', 'SFGate', 'large'),
    'mercurynews.com':          ('CA', 'San Jose Mercury News', 'large'),
    'sandiegouniontribune.com': ('CA', 'San Diego Union-Tribune', 'large'),
    'sacbee.com':               ('CA', 'Sacramento Bee', 'medium'),
    'fresnobee.com':            ('CA', 'Fresno Bee', 'medium'),
    'ocregister.com':           ('CA', 'Orange County Register', 'medium'),
    'oregonlive.com':           ('OR', 'The Oregonian (Portland)', 'large'),
    'statesmanjournal.com':     ('OR', 'Salem Statesman Journal', 'small'),
    'registerguard.com':        ('OR', 'Eugene Register-Guard', 'small'),
    'seattletimes.com':         ('WA', 'Seattle Times', 'large'),
    'spokesman.com':            ('WA', 'Spokane Spokesman-Review', 'medium'),
    'thenewstribune.com':       ('WA', 'Tacoma News Tribune', 'medium'),
    'azcentral.com':            ('AZ', 'Arizona Republic (Phoenix)', 'large'),
    'tucson.com':               ('AZ', 'Arizona Daily Star (Tucson)', 'medium'),
    'denverpost.com':           ('CO', 'Denver Post', 'large'),
    'gazette.com':              ('CO', 'Colorado Springs Gazette', 'medium'),
    'sltrib.com':               ('UT', 'Salt Lake Tribune', 'medium'),
    'deseret.com':              ('UT', 'Deseret News', 'medium'),
    'reviewjournal.com':        ('NV', 'Las Vegas Review-Journal', 'large'),
    'rgj.com':                  ('NV', 'Reno Gazette Journal', 'small'),
    'abqjournal.com':           ('NM', 'Albuquerque Journal', 'medium'),
    'lcsun-news.com':           ('NM', 'Las Cruces Sun-News', 'small'),
    'idahostatesman.com':       ('ID', 'Idaho Statesman (Boise)', 'medium'),
    'billingsgazette.com':      ('MT', 'Billings Gazette', 'small'),
    'missoulian.com':           ('MT', 'Missoulian', 'small'),
    'trib.com':                 ('WY', 'Casper Star-Tribune', 'small'),
    'wyomingnews.com':          ('WY', 'Wyoming Tribune Eagle (Cheyenne)', 'small'),
    'adn.com':                  ('AK', 'Anchorage Daily News', 'medium'),
    'staradvertiser.com':       ('HI', 'Honolulu Star-Advertiser', 'medium'),
    'delawareonline.com':       ('DE', 'Wilmington News Journal', 'small'),
}

# Quick stats
from collections import Counter

states_covered = Counter(state for state, _, _ in US_NEWSPAPER_DOMAINS.values())
sizes = Counter(size for _, _, size in US_NEWSPAPER_DOMAINS.values())

print(f"Total domains: {len(US_NEWSPAPER_DOMAINS)}")
print(f"Unique states covered: {len(states_covered)}")
print(f"\nDomains per state (top 10):")
for state, count in states_covered.most_common(10):
    print(f"  {state}: {count}")
print(f"\nDomains per state (bottom 10):")
for state, count in states_covered.most_common()[-10:]:
    print(f"  {state}: {count}")
print(f"\nSize distribution:")
for size, count in sizes.most_common():
    print(f"  {size}: {count}")

# States NOT covered yet
all_states = set(['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID',
                  'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS',
                  'MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
                  'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV',
                  'WI','WY','DC'])
covered = set(states_covered.keys())
missing = all_states - covered
print(f"\nStates not covered: {missing if missing else 'NONE — all 50 states + DC covered'}")

Total domains: 121
Unique states covered: 51

Domains per state (top 10):
  CA: 8
  NY: 7
  TX: 7
  FL: 6
  PA: 4
  IL: 4
  OH: 4
  TN: 4
  MA: 3
  NJ: 3

Domains per state (bottom 10):
  NE: 1
  KS: 1
  SD: 1
  ND: 1
  WV: 1
  AR: 1
  ID: 1
  AK: 1
  HI: 1
  DE: 1

Size distribution:
  medium: 55
  large: 41
  small: 22
  national: 3

States not covered: NONE — all 50 states + DC covered


In [ ]:
# Extract GDELT GKG records for January 2024
# Filter to articles from our US newspaper domain list

# Build the SQL IN clause from our domain list
domain_list = list(US_NEWSPAPER_DOMAINS.keys())
domain_quoted = ", ".join(f"'{d}'" for d in domain_list)

# Build query
query = f"""
SELECT
    DATE(_PARTITIONTIME) as date,
    GKGRECORDID as gkg_record_id,
    SourceCommonName as source_domain,
    DocumentIdentifier as article_url,
    V2Themes as themes,
    V2Tone as tone,
    V2Locations as locations,
    V2Persons as persons,
    V2Organizations as organizations,
    DATE as article_date
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2024-01-01")
  AND _PARTITIONTIME < TIMESTAMP("2024-02-01")
  AND source_domain IN ({domain_quoted})
"""

print("Query length (chars):", len(query))
print("\nFirst 500 chars of query:")
print(query[:500])

# Dry-run first to check how much data this will scan
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = client.query(query, job_config=job_config)
bytes_scanned = dry_run_job.total_bytes_processed
gb_scanned = bytes_scanned / (1024**3)
print(f"\nDry run: this query will scan {gb_scanned:.2f} GB")
print(f"Sandbox free tier limit: 1 TB/month")

Query length (chars): 2539

First 500 chars of query:

SELECT
    DATE(_PARTITIONTIME) as date,
    GKGRECORDID,
    SourceCommonName,
    DocumentIdentifier,
    V2Themes,
    V2Tone,
    V2Locations,
    V2Persons,
    V2Organizations,
    DATE
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2024-01-01")
  AND _PARTITIONTIME < TIMESTAMP("2024-02-01")
  AND SourceCommonName IN ('nytimes.com', 'nypost.com', 'nydailynews.com', 'newsday.com', 'buffalonews.com', 'syracuse.com', 'democratandchronicle.com', 'washingtonpost.com

Dry run: this query will scan 18.51 GB
Sandbox free tier limit: 1 TB/month


In [4]:
# Run the actual query and save results
import time

OUTPUT_DIR = Path(r"D:\Shikha\bias_project\corpus\gdelt\2024-01")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "gdelt_articles.csv"

print(f"Running query (this will take 30-90 seconds)...")
start_time = time.time()

# Run the actual query
job_config = bigquery.QueryJobConfig(use_query_cache=True)
query_job = client.query(query, job_config=job_config)
df = query_job.to_dataframe()

elapsed = time.time() - start_time
bytes_billed = query_job.total_bytes_billed / (1024**3)

print(f"\nQuery completed in {elapsed:.1f} seconds")
print(f"Bytes billed: {bytes_billed:.2f} GB")
print(f"Rows returned: {len(df):,}")

# Add state mapping
print(f"\nMapping domains to states...")
df['state'] = df['SourceCommonName'].map(lambda d: US_NEWSPAPER_DOMAINS.get(d, ('UNKNOWN', '', ''))[0])
df['source_name'] = df['SourceCommonName'].map(lambda d: US_NEWSPAPER_DOMAINS.get(d, ('', '', ''))[1])
df['source_size'] = df['SourceCommonName'].map(lambda d: US_NEWSPAPER_DOMAINS.get(d, ('', '', ''))[2])

# Save to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")
print(f"File size: {OUTPUT_CSV.stat().st_size / (1024**2):.1f} MB")

Running query (this will take 30-90 seconds)...


d:\Shikha\bias_project\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Query completed in 598.6 seconds
Bytes billed: 18.51 GB
Rows returned: 97,280

Mapping domains to states...

Saved to: D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv
File size: 240.4 MB


In [5]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path(r"D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv")

# Load
df = pd.read_csv(CSV_PATH)

# Rename columns for clarity
df = df.rename(columns={
    'DocumentIdentifier': 'article_url',
    'SourceCommonName': 'source_domain',
    'GKGRECORDID': 'gkg_record_id',
    'V2Themes': 'themes',
    'V2Tone': 'tone',
    'V2Locations': 'locations',
    'V2Persons': 'persons',
    'V2Organizations': 'organizations',
    'DATE': 'article_date',
})

# Save back
df.to_csv(CSV_PATH, index=False)

print(f"Renamed columns in {CSV_PATH}")
print(f"\nNew columns: {list(df.columns)}")
print(f"Total rows: {len(df):,}")

Renamed columns in D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv

New columns: ['date', 'gkg_record_id', 'source_domain', 'article_url', 'themes', 'tone', 'locations', 'persons', 'organizations', 'DATE_1', 'state', 'source_name', 'source_size']
Total rows: 97,280


In [6]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path(r"D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv")

df = pd.read_csv(CSV_PATH)

df = df.rename(columns={'DATE_1': 'article_date'})

df.to_csv(CSV_PATH, index=False)

print(f"Columns now: {list(df.columns)}")
print(f"Total rows: {len(df):,}")

Columns now: ['date', 'gkg_record_id', 'source_domain', 'article_url', 'themes', 'tone', 'locations', 'persons', 'organizations', 'article_date', 'state', 'source_name', 'source_size']
Total rows: 97,280


In [7]:
import pandas as pd

df = pd.read_csv(r"D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv")

print(f"Total articles: {len(df):,}")
print(f"Unique domains: {df['source_domain'].nunique()}")
print(f"Unique states:  {df['state'].nunique()}")
print(f"File size:      ~240 MB")

print(f"\n--- Articles per state (top 10) ---")
print(df['state'].value_counts().head(10))

print(f"\n--- Articles per state (bottom 10) ---")
print(df['state'].value_counts().tail(10))

print(f"\n--- Articles per source (top 15) ---")
print(df['source_domain'].value_counts().head(15))

print(f"\n--- Domains with ZERO hits in this month ---")
domains_with_data = set(df['source_domain'].unique())
expected_domains = set(US_NEWSPAPER_DOMAINS.keys())
missing = expected_domains - domains_with_data
print(f"Total missing: {len(missing)}/{len(expected_domains)}")
if missing:
    for d in sorted(missing)[:15]:
        state, name, _ = US_NEWSPAPER_DOMAINS[d]
        print(f"  {d:30s} ({state}: {name})")

print(f"\n--- Sample article URLs ---")
for url in df['article_url'].sample(min(5, len(df))).values:
    print(f"  {url[:100]}")

Total articles: 97,280
Unique domains: 83
Unique states:  44
File size:      ~240 MB

--- Articles per state (top 10) ---
state
NY    14344
CA    12370
MA     5952
MN     4379
TX     4184
PA     4038
CO     3600
FL     3572
DC     2967
IL     2673
Name: count, dtype: int64

--- Articles per state (bottom 10) ---
state
CT    648
OK    623
WV    478
AZ    473
ID    453
NM    420
ND    390
KS     90
MS     60
KY     40
Name: count, dtype: int64

--- Articles per source (top 15) ---
source_domain
nypost.com                  4879
nytimes.com                 3662
startribune.com             3639
washingtonpost.com          2967
sandiegouniontribune.com    2831
bostonglobe.com             2693
mlive.com                   2653
arkansasonline.com          2602
gazette.com                 2480
latimes.com                 2176
mercurynews.com             2077
nj.com                      2014
masslive.com                1987
pennlive.com                1937
newsday.com                 1860
Name: c

In [8]:
covered_states = set(df['state'].unique())
all_us_states = set(['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID',
                     'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS',
                     'MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
                     'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV',
                     'WI','WY','DC'])
missing_states = all_us_states - covered_states
print(f"States with NO articles: {missing_states}")
print(f"Count: {len(missing_states)}")

States with NO articles: {'RI', 'VT', 'TN', 'IA', 'IN', 'DE', 'SD'}
Count: 7


In [9]:
# Search GDELT for domains containing state-named newspaper substrings
test_query = """
SELECT DISTINCT SourceCommonName, COUNT(*) as article_count
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2024-01-01")
  AND _PARTITIONTIME < TIMESTAMP("2024-02-01")
  AND (
    SourceCommonName LIKE '%tennessean%' OR
    SourceCommonName LIKE '%desmoines%' OR
    SourceCommonName LIKE '%indystar%' OR
    SourceCommonName LIKE '%burlington%' OR
    SourceCommonName LIKE '%providence%' OR
    SourceCommonName LIKE '%argus%' OR
    SourceCommonName LIKE '%delaware%' OR
    SourceCommonName LIKE '%jconline%' OR
    SourceCommonName LIKE '%commercialappeal%' OR
    SourceCommonName LIKE '%knoxnews%' OR
    SourceCommonName LIKE '%courier-journal%'
  )
GROUP BY SourceCommonName
ORDER BY article_count DESC
"""

print("Searching GDELT for variant domains of missing-state papers...")
result = client.query(test_query).to_dataframe()
print(f"\nFound {len(result)} matching domains in GDELT:")
print(result.to_string(index=False))

Searching GDELT for variant domains of missing-state papers...

Found 24 matching domains in GDELT:
          SourceCommonName  article_count
     southwalesargus.co.uk           2341
thetelegraphandargus.co.uk           2109
            theargus.co.uk           1994
     singletonargus.com.au           1945
       macleayargus.com.au           1755
        delawarepublic.org            785
 jacksonprogress-argus.com            622
             newsargus.com            516
            timesargus.com            248
     burlington-record.com            175
         argusobserver.com             97
    townsquaredelaware.com             82
           argus-press.com             67
              delaware.gov             42
         wausharaargus.com             34
    condobolinargus.com.au             31
      providenceonline.com             28
   delawarebusinessnow.com             26
   goondiwindiargus.com.au             25
            wdjconline.com              7
             bjcon

In [10]:
# Add non-Gannett alternatives for the 7 missing states
MISSING_STATE_DOMAINS = {
    # Rhode Island
    'golocalprov.com':       ('RI', 'GoLocalProv', 'medium'),
    'wpri.com':              ('RI', 'WPRI Eyewitness News', 'medium'),
    'whatsupnewp.com':       ('RI', "What's Up Newp", 'small'),
    
    # Vermont
    'vtdigger.org':          ('VT', 'VTDigger', 'medium'),
    'sevendaysvt.com':       ('VT', 'Seven Days', 'medium'),
    'mychamplainvalley.com': ('VT', 'WFFF/WVNY', 'small'),
    
    # Tennessee
    'wpln.org':              ('TN', 'WPLN Nashville Public Radio', 'medium'),
    'newschannel5.com':      ('TN', 'NewsChannel 5 Nashville', 'medium'),
    'tnlookout.com':         ('TN', 'Tennessee Lookout', 'medium'),
    'wbir.com':              ('TN', 'WBIR Knoxville', 'medium'),
    
    # Iowa
    'iowapublicradio.org':   ('IA', 'Iowa Public Radio', 'medium'),
    'kcci.com':              ('IA', 'KCCI Des Moines', 'medium'),
    'iowacapitaldispatch.com': ('IA', 'Iowa Capital Dispatch', 'medium'),
    'thegazette.com':        ('IA', 'Cedar Rapids Gazette', 'medium'),
    
    # Indiana
    'wfyi.org':              ('IN', 'WFYI Indianapolis', 'medium'),
    'indianapublicmedia.org':('IN', 'Indiana Public Media', 'medium'),
    'wishtv.com':            ('IN', 'WISH-TV Indianapolis', 'medium'),
    'indianacapitalchronicle.com': ('IN', 'Indiana Capital Chronicle', 'medium'),
    
    # Delaware
    'spotlightdelaware.org': ('DE', 'Spotlight Delaware', 'medium'),
    'wdde.org':              ('DE', 'Delaware Public Media', 'medium'),
    'wmdt.com':              ('DE', 'WMDT-TV', 'small'),
    
    # South Dakota
    'keloland.com':          ('SD', 'KELOLAND Media (Sioux Falls)', 'medium'),
    'sdpb.org':              ('SD', 'South Dakota Public Broadcasting', 'medium'),
    'rapidcityjournal.com':  ('SD', 'Rapid City Journal', 'medium'),
    'dakotanewsnow.com':     ('SD', 'Dakota News Now', 'small'),
}

print(f"Adding {len(MISSING_STATE_DOMAINS)} new domains for {len(set(s for s,_,_ in MISSING_STATE_DOMAINS.values()))} missing states")

# Combine with the original mapping
ALL_DOMAINS = {**US_NEWSPAPER_DOMAINS, **MISSING_STATE_DOMAINS}
print(f"Combined mapping now: {len(ALL_DOMAINS)} domains")

# Query GDELT for just these new domains
new_domain_list = list(MISSING_STATE_DOMAINS.keys())
new_domain_quoted = ", ".join(f"'{d}'" for d in new_domain_list)

gap_query = f"""
SELECT
    DATE(_PARTITIONTIME) as date,
    GKGRECORDID as gkg_record_id,
    SourceCommonName as source_domain,
    DocumentIdentifier as article_url,
    V2Themes as themes,
    V2Tone as tone,
    V2Locations as locations,
    V2Persons as persons,
    V2Organizations as organizations,
    DATE as article_date
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2024-01-01")
  AND _PARTITIONTIME < TIMESTAMP("2024-02-01")
  AND SourceCommonName IN ({new_domain_quoted})
"""

# Dry run first
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = client.query(gap_query, job_config=job_config)
gb_scanned = dry_run_job.total_bytes_processed / (1024**3)
print(f"\nDry run: gap-fill query will scan {gb_scanned:.2f} GB")

Adding 25 new domains for 7 missing states
Combined mapping now: 146 domains

Dry run: gap-fill query will scan 18.51 GB


In [11]:
# Execute the gap-fill query
import time
from pathlib import Path

print("Running gap-fill query (will take a few minutes due to REST API download)...")
start_time = time.time()

job_config = bigquery.QueryJobConfig(use_query_cache=True)
gap_job = client.query(gap_query, job_config=job_config)
gap_df = gap_job.to_dataframe()

elapsed = time.time() - start_time
print(f"\nQuery completed in {elapsed:.1f} seconds")
print(f"Rows returned: {len(gap_df):,}")

if len(gap_df) > 0:
    # Add state mapping using the combined dictionary
    gap_df['state'] = gap_df['source_domain'].map(
        lambda d: ALL_DOMAINS.get(d, ('UNKNOWN', '', ''))[0]
    )
    gap_df['source_name'] = gap_df['source_domain'].map(
        lambda d: ALL_DOMAINS.get(d, ('', '', ''))[1]
    )
    gap_df['source_size'] = gap_df['source_domain'].map(
        lambda d: ALL_DOMAINS.get(d, ('', '', ''))[2]
    )
    
    # Rename DATE_1 if it appears
    if 'DATE_1' in gap_df.columns:
        gap_df = gap_df.rename(columns={'DATE_1': 'article_date'})
    
    # Show what we got per state
    print(f"\nNew articles per state:")
    print(gap_df['state'].value_counts())
    
    print(f"\nNew articles per source:")
    print(gap_df['source_domain'].value_counts())
    
    # Domains that returned ZERO
    queried_domains = set(MISSING_STATE_DOMAINS.keys())
    found_domains = set(gap_df['source_domain'].unique())
    still_missing = queried_domains - found_domains
    print(f"\nDomains with ZERO hits: {len(still_missing)}")
    for d in sorted(still_missing):
        state, name, _ = MISSING_STATE_DOMAINS[d]
        print(f"  {d:30s} ({state}: {name})")
    
    # Append to existing CSV
    CSV_PATH = Path(r"D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv")
    original_df = pd.read_csv(CSV_PATH)
    print(f"\nOriginal CSV: {len(original_df):,} rows")
    
    # Combine and save
    combined_df = pd.concat([original_df, gap_df], ignore_index=True)
    combined_df.to_csv(CSV_PATH, index=False)
    print(f"Combined CSV: {len(combined_df):,} rows (added {len(gap_df):,})")
    print(f"File size: {CSV_PATH.stat().st_size / (1024**2):.1f} MB")
else:
    print("\nNo new articles found for any of the additional domains.")
    print("All 7 states remain uncovered in GDELT for January 2024.")

Running gap-fill query (will take a few minutes due to REST API download)...

Query completed in 13.3 seconds
Rows returned: 6,712

New articles per state:
state
IA    2085
SD    1219
TN    1090
IN     894
VT     548
DE     442
RI     434
Name: count, dtype: int64

New articles per source:
source_domain
kcci.com                       1041
wishtv.com                      652
wbir.com                        627
thegazette.com                  585
dakotanewsnow.com               550
newschannel5.com                463
iowapublicradio.org             445
wmdt.com                        442
rapidcityjournal.com            299
vtdigger.org                    285
sdpb.org                        264
wfyi.org                        219
sevendaysvt.com                 206
golocalprov.com                 205
wpri.com                        126
keloland.com                    106
whatsupnewp.com                 103
mychamplainvalley.com            57
indianacapitalchronicle.com      17
iowacapital